# DUUI-Py Evaluation: Python Annotator Ergonomics and Async Codec Experiments

Master's computer science research project presentation, May 2026.

This notebook is written for a technical review audience. It summarizes the motivation, implementation strategy, evaluation setup, measured results, and limitations of DUUI-Py. The evaluation terminology is intentionally restricted to **baseline** and **async variant**. Raw service or file names are kept only where needed for provenance.


## 1. Motivation

DUUI annotators normally require more than model code. A working annotator must expose DUUI-compatible endpoints, provide metadata and documentation, package a UIMA type system, translate CAS data into the annotator request, translate annotator output back into CAS annotations, and provide a Lua communication layer. This is repetitive integration work rather than annotator logic.

DUUI-Py was built to reduce this boilerplate for Python-based annotators. Python is already common in NLP research and prototyping, so the project keeps annotator authors in Python while the framework handles the DUUI endpoint surface, UIMA model classes, descriptors, codecs, adapters, logging, and telemetry.

The central engineering question is whether this abstraction can remain cheap inside `/v1/process`. The evaluation therefore compares **baseline** annotators against **async variants** that use the generated MsgPack/Lua codec and async adapter path.


## 2. System Overview

The verified DUUI-Py application factory constructs the annotator configuration, resolves the codec, selects the request adapter, and registers the DUUI endpoint surface. The framework provides `/v1/typesystem`, `/v1/communication_layer`, `/v1/details/input_output`, `/v1/documentation`, `/v1/process`, and `/v2/events`.

Annotator authors supply a descriptor and a process function. Generated UIMA classes are available for annotation outputs, so annotator code can emit typed objects instead of hand-written transport dictionaries. This is important both for ergonomics and for generated codec performance, because the descriptor and type model provide type names, feature names, ranges, and reference metadata.

The design constraint is that annotators should not contain wire-format helpers or local UIMA model shims. The codec and adapter are framework concerns; the annotator should express the NLP procedure and declared UIMA inputs/outputs.


## 3. Events and Telemetry

DUUI-Py adds a `v2/events` endpoint for structured logs, metrics, spans, summaries, and errors. The endpoint is streamed independently of the process response, so operational data does not have to be packed into the CAS output.

The checked telemetry implementation uses request-scoped identifiers, severity levels, metric data points, resource attributes, and histogram summaries. This made the later bottleneck analysis possible: saved runs include request duration, response receive/decode, request decode, payload bytes, event counts, CPU, and memory where the evaluation preserved those data.

Example annotator-side usage remains local and small:

```python
with telemetry.timer("annotator_processing_ms"):
    output = run_model(request)

telemetry.metric("candidate_count", len(candidates), unit="count")
telemetry.log("debug", "candidate generation completed", candidates=len(candidates))
```


## 4. Codec and Adapter Design

DUUI-Py supports a backward-compatible path for annotators with existing custom Lua scripts. That path is useful for migration because it preserves older request and response contracts.

The experimental path is descriptor-generated. The MsgPack/Lua codec derives Lua serialization and deserialization logic from descriptor and type-system metadata. The async adapter then streams request decoding and response encoding in chunks when the codec supports it.

The practical idea is to separate stable schema-like information from instance data. Type names, feature names, primitive ranges, reference-valued fields, and output projections come from the descriptor and type system. Individual chunks should then mostly contain annotation instance values. This only helps when the selected descriptor and active process strategy avoid sending data that is not needed.


## 5. Evaluation Setup

The evaluation uses preserved local artifacts only. Numeric claims are not reconstructed from code.

| Annotator | Preserved data used here | What is measured |
|---|---|---|
| spaCy | `/tmp/spacy-scale-windowed.log`, `/tmp/spacy-scale-direct-inlineref.log` | Buffered baseline vs async variant under increasing generated annotation counts. |
| TaxoNERD | `examples/taxonerd_big_evaluation_report.md`, `examples/taxonerd_actual_legacy_baseline_comparison.md`, `examples/taxonerd_correctness_eval_results.csv` | Broad XMI latency, payloads, process substeps, linker timings, and correctness-run latency. |
| Gazetteer | `examples/gazetteer_legacy_vs_msgpack_evaluation_results.json` | Paired baseline vs async variant on XMI-derived chunks plus event/resource telemetry. |
| GNFinder | Implementation files and Java matrix test XML | Harness and implementation evidence only; row-level timing output was not preserved. |

The primary metric is process latency. Secondary metrics include p95/p99 tail latency, request and response bytes, response receive/decode, request decode, throughput derived from latency, CPU, and memory where available.


In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("reports/data")
spacy = pd.read_csv(DATA / "spacy_transport_scale.csv")
taxonerd_overall = pd.read_csv(DATA / "taxonerd_overall_latency.csv")
taxonerd_process = pd.read_csv(DATA / "taxonerd_transport_process_medians.csv")
taxonerd_actual = pd.read_csv(DATA / "taxonerd_actual_legacy_pair.csv")
gazetteer = pd.read_csv(DATA / "gazetteer_rows.csv")
gazetteer_pairs = pd.read_csv(DATA / "gazetteer_pairs.csv")
gazetteer_metrics = pd.read_csv(DATA / "gazetteer_metric_points.csv")

spacy.head(), taxonerd_overall.head(), gazetteer.head()


## 6. Results: spaCy

spaCy is the densest evaluated annotator. It can emit tokens, lemmas, POS tags, morphology, dependencies, sentences, and named entities. This makes response-side annotation application large enough for async deserialization to matter.


<figure>
  <img src="figures/spacy_transport_latency.png" style="width:760px; max-width:100%; height:auto; display:block; margin:0 auto;">
  <figcaption style="font-size:0.92em; margin-top:0.6em;"><strong>Figure 1. spaCy latency for baseline vs async variant as generated annotation count increases. At 78,000 annotations, the saved scale logs show improvements from 2,879 ms to 2,530 ms and from 4,112 ms to 3,353 ms. Smaller cases are mixed because fixed overhead dominates.</strong></figcaption>
</figure>


<figure>
  <img src="figures/spacy_transport_components.png" style="width:760px; max-width:100%; height:auto; display:block; margin:0 auto;">
  <figcaption style="font-size:0.92em; margin-top:0.6em;"><strong>Figure 2. spaCy procedural measurements. Async helps when response application work is large enough to overlap with response production; it is not a universal win for small outputs.</strong></figcaption>
</figure>


## 7. Results: TaxoNERD

TaxoNERD combines NER and taxonomy linking. In the broad saved run, both baseline and async containers used `en_ner_eco_md` with `gbif_backbone`; the saved report explicitly states that no German TaxoNERD LIVB model was available in both containers. These numbers are therefore a same-model transport/procedure comparison, not a final German-model quality claim.

The broad run covers 45 XMI/GZip documents with 854,555 input characters, 10,602 sentences, 302,616 tokens, and 7,682 named entities.


<figure>
  <img src="figures/taxonerd_latency_tail.png" style="width:760px; max-width:100%; height:auto; display:block; margin:0 auto;">
  <figcaption style="font-size:0.92em; margin-top:0.6em;"><strong>Figure 3. TaxoNERD latency and tail latency. The whole-document async variant has a slightly lower median than baseline, 592 ms versus 608 ms, but worse p95 and p99 latency. Span-based async variants are slower and alter output counts.</strong></figcaption>
</figure>


<figure>
  <img src="figures/taxonerd_process_costs.png" style="width:760px; max-width:100%; height:auto; display:block; margin:0 auto;">
  <figcaption style="font-size:0.92em; margin-top:0.6em;"><strong>Figure 4. TaxoNERD process-request substeps. The async variant lowers reported request duration in the whole-document case, but request decode and response decode become non-negligible.</strong></figcaption>
</figure>


<figure>
  <img src="figures/taxonerd_payload.png" style="width:760px; max-width:100%; height:auto; display:block; margin:0 auto;">
  <figcaption style="font-size:0.92em; margin-top:0.6em;"><strong>Figure 5. TaxoNERD payload sizes. The saved report records about 9.8 KB median request payload for baseline and about 91 KB for async variants. This is the clearest measured process-request overhead.</strong></figcaption>
</figure>


<figure>
  <img src="figures/taxonerd_linker_ann.png" style="width:740px; max-width:100%; height:auto; display:block; margin:0 auto;">
  <figcaption style="font-size:0.92em; margin-top:0.6em;"><strong>Figure 6. TaxoNERD linker optimization inside the async variant. Batching reduces ANN lookup from 748.2 ms to 81.7 ms for the same 220 found and linked outputs.</strong></figcaption>
</figure>


<figure>
  <img src="figures/taxonerd_correctness_distribution.png" style="width:620px; max-width:100%; height:auto; display:block; margin:0 auto;">
  <figcaption style="font-size:0.92em; margin-top:0.6em;"><strong>Figure 7. TaxoNERD correctness-run latency across 12 XMI documents. The run found and linked 68 taxon annotations with no failed documents; linker timing fields were zero in this saved CSV.</strong></figcaption>
</figure>


The TaxoNERD result is the main cautionary result. Async transport alone is not enough if the request path sends structural CAS input that the active whole-document strategy does not use. The useful optimization was semantic-preserving linker batching, not arbitrary span splitting or replacing the linker path.


## 8. Results: Gazetteer

Gazetteer has a different cost profile. The backend is native and fast, so fixed wrapper overhead is visible. The saved evaluation contains two resource subsets and six XMI-derived chunks per subset. Paired rows are output-equivalent.


<figure>
  <img src="figures/gazetteer_pair_latency.png" style="width:720px; max-width:100%; height:auto; display:block; margin:0 auto;">
  <figcaption style="font-size:0.92em; margin-top:0.6em;"><strong>Figure 8. Gazetteer paired latency. Baseline and async variant are both in the low single-digit millisecond range; async does not show a speedup because the workload is too small to hide meaningful response-application work.</strong></figcaption>
</figure>


<figure>
  <img src="figures/gazetteer_throughput.png" style="width:660px; max-width:100%; height:auto; display:block; margin:0 auto;">
  <figcaption style="font-size:0.92em; margin-top:0.6em;"><strong>Figure 9. Gazetteer throughput derived from latency. The throughput result mirrors the latency result: fixed wrapper overhead dominates this tiny native-backend workload.</strong></figcaption>
</figure>


<figure>
  <img src="figures/gazetteer_telemetry_events.png" style="width:660px; max-width:100%; height:auto; display:block; margin:0 auto;">
  <figcaption style="font-size:0.92em; margin-top:0.6em;"><strong>Figure 10. Gazetteer telemetry volume. Each resource subset preserved 728 events: 672 metric events, 28 log events, 14 span events, and 14 summary events.</strong></figcaption>
</figure>


<figure>
  <img src="figures/gazetteer_resource_metrics.png" style="width:720px; max-width:100%; height:auto; display:block; margin:0 auto;">
  <figcaption style="font-size:0.92em; margin-top:0.6em;"><strong>Figure 11. Gazetteer CPU and memory samples. CPU is coarse at millisecond scale; RSS is more stable. Dedicated network telemetry points were not preserved in this artifact.</strong></figcaption>
</figure>


## 9. GNFinder Evidence Limit

GNFinder has both baseline and async implementations in the repository, and the async implementation calls the real `gnfinder` executable before emitting generated UIMA classes. The Java matrix test includes a GNFinder baseline-versus-async comparison, and the surefire XML records the GNFinder test case as passed with total test time 56.076 seconds.

However, row-level baseline and async timings were not preserved in a CSV, JSON report, markdown report, or surefire stdout artifact. For that reason, this notebook does not plot GNFinder as a primary numeric result.


## 10. Bottlenecks

| Annotator | Measured bottleneck | Consequence |
|---|---|---|
| spaCy | Dense response-side annotation application | Async variant helps at large annotation counts. |
| TaxoNERD | Unnecessary async request payload plus linker lookup cost | Whole-document async barely improves median and worsens tail; linker batching helps but does not fully solve end-to-end latency. |
| Gazetteer | Fixed Python/DUUI wrapper overhead around a very fast backend | Async transport is not the right optimization lever for tiny outputs. |
| GNFinder | Row-level timing missing | Implementation exists, but the saved data is insufficient for bottleneck attribution. |

The general conclusion is that descriptor-generated async transport helps when there is enough response-side work to overlap. It does not compensate for over-broad request projection or annotator procedure bottlenecks.


## 11. Limitations

The evaluation is based on preserved artifacts from an active implementation phase. Some annotators have detailed row-level data, while GNFinder does not. TaxoNERD used the same available English ecological model in both containers, so it is a transport/procedure comparison rather than a final German taxonomic NER quality evaluation. Gazetteer used small fast resource subsets rather than the full BioFID configuration. CPU samples are coarse for millisecond-scale requests, and dedicated network telemetry points were not preserved uniformly.


## 12. Conclusion

DUUI-Py successfully centralizes DUUI annotator boilerplate: endpoint routing, descriptors, generated UIMA classes, custom Lua migration, generated MsgPack/Lua communication, async adapters, and streamed telemetry.

The performance conclusion is workload-specific. The async variant is useful for dense response workloads such as spaCy at high annotation counts. TaxoNERD shows that descriptor projection must be precise, because avoidable request structure can erase transport gains. Gazetteer shows that fixed wrapper overhead dominates very small native-backend calls. The next technical focus should therefore be lower-footprint generated data structures and descriptor-accurate input projection, while keeping codec and adapter logic out of annotator-specific code.


## References

- UC San Diego Department of Psychology. "Research Paper Structure." <https://psychology.ucsd.edu/undergraduate-program/undergraduate-resources/academic-writing-resources/writing-research-papers/research-paper-structure.html>
- UBC Science Writing Resources. "IMRAD (Introduction, Methods, Results and Discussion)." <https://scwrl.ubc.ca/stem-writing-resources/features-of-academic-stem-research-writing/imrad/>
- MessagePack. <https://msgpack.org/>
- OpenTelemetry Logs Data Model. <https://opentelemetry.io/docs/specs/otel/logs/data-model/>
- OpenTelemetry Metrics Data Model. <https://opentelemetry.io/docs/specs/otel/metrics/data-model/>
- MDN Web Docs. "Using server-sent events." <https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events>
- Oracle Java SE 21 API, `HttpRequest.BodyPublishers` and `HttpResponse.BodySubscribers`. <https://docs.oracle.com/en/java/javase/21/docs/api/java.net.http/java/net/http/HttpRequest.BodyPublishers.html>, <https://docs.oracle.com/en/java/javase/21/docs/api/java.net.http/java/net/http/HttpResponse.BodySubscribers.html>
- Dean and Barroso. "The Tail at Scale." <https://research.google/pubs/the-tail-at-scale/>
- seaborn documentation for boxplots and empirical distribution plots. <https://seaborn.pydata.org/generated/seaborn.boxplot.html>, <https://seaborn.pydata.org/generated/seaborn.ecdfplot.html>


## Appendix: Provenance

| Area | Local source |
|---|---|
| Application endpoints | `src/duui_py/app.py:65-210` |
| Descriptor/config model | `src/duui_py/models/config.py:62-254` |
| Telemetry | `src/duui_py/telemetry.py:31-361` |
| Adapters | `src/duui_py/adapters.py:52-673` |
| MsgPack codec | `src/duui_py/codecs/msgpack_lua/codec.py:39-249` |
| Descriptor wire plan | `src/duui_py/codecs/msgpack_lua/wire.py:46-255` |
| Events documentation | `docs/events.md:3-126` |
| MsgPack documentation | `docs/msgpack-lua.md:1-84` |
| spaCy examples | `examples/spacy-lua-msgpack/spacy_annotator.py`, `examples/spacy-legacy-lua/spacy_legacy_annotator.py` |
| TaxoNERD examples and reports | `examples/taxonerd-msgpack-lua/taxonerd_annotator.py`, `examples/taxonerd_*` |
| Gazetteer example and report | `examples/gazetteer-msgpack-lua/gazetteer_annotator.py`, `examples/gazetteer_legacy_vs_msgpack_evaluation_results.json` |
| GNFinder example | `examples/gnfinder-msgpack-lua/gnfinder_annotator.py` |
| Extracted data | `reports/data/*.csv` |
| Static figures | `reports/figures/*.png` |
